# 07 — Pre-action passing-line geometry (all 20 matches)

**Goal:** Describe contemporaneous straight-line geometry between each ball carrier and each teammate. This is **not** a pass feasibility model or a counterfactual evaluation. A static line without nearby defender coordinates does not establish pass completion, arrival timing, available passing routes, or eventual outcomes.

**Inputs:** Notebook 05's `outputs/P1_canonical_decisions_match_*_ROLES.jsonl` (not future outcomes). **Outputs:** `outputs/P2_pass_lane_geometry_gate.json` and `outputs/P2_pass_lane_candidates_match_{id}.jsonl`.

For every carrier–teammate pair, calculate carrier–teammate distance and closest opponent to the **open line segment** from carrier to teammate. A defender located near either endpoint is recorded but does not on its own constitute a line interception. The minimum is a geometric diagnostic, not a travel-time or passing-feasibility assessment. Extrapolated coordinates are kept separate from detected-only sensitivity calculations. No forward-direction labels yet.

In [1]:
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
import json,math,statistics
ROOT=next((p for p in (Path.cwd().resolve(),*Path.cwd().resolve().parents) if (p/'data'/'matches').is_dir()),None)
if ROOT is None: raise FileNotFoundError('Run from the SkillCorner repository.')
OUT=ROOT/'outputs';OUT.mkdir(exist_ok=True)
FILES=sorted(OUT.glob('P1_canonical_decisions_match_*_ROLES.jsonl'))
assert len(FILES)==20,f'Expected 20 canonical files, found {len(FILES)}'
print('Matches:',len(FILES))

Matches: 20


## Geometry and explicit availability rules

A valid source coordinate must be finite. A passing-line candidate requires a carrier position and teammate position at the **same observed decision frame**. With no opponent coordinates, separation is `null`—never infinite or unobstructed. For zero-length segments the geometric diagnostic is undefined. Neither team orientation nor event-derived passing options are used to certify availability.

In [2]:
def valid(p):
    return isinstance(p,(list,tuple)) and len(p)==2 and all(isinstance(x,(int,float)) and math.isfinite(x) for x in p)
def distance(a,b):return math.hypot(a[0]-b[0],a[1]-b[1])
def segment_margin(a,b,p):
    # perpendicular distance to *interior* segment only; endpoints do not count as interior obstruction.
    dx,dy=b[0]-a[0],b[1]-a[1]
    sq=dx*dx+dy*dy
    if sq<=1e-12:return None
    t=((p[0]-a[0])*dx+(p[1]-a[1])*dy)/sq
    if not (0<t<1):return None
    return (math.hypot(p[0]-(a[0]+t*dx),p[1]-(a[1]+t*dy)),t)
def actors(r,role,include_extrapolated):
    qualities=('detected','extrapolated') if include_extrapolated else ('detected',)
    return [(p.get('player_id'),p.get('xy'),quality) for quality in qualities
            for p in r.get(f'{role}_{quality}',[]) if valid(p.get('xy'))]
def clearance(a,b,opponents):
    if not opponents:return None,None
    interior=[(v[0],pid) for pid,pos,_ in opponents if (v:=segment_margin(a,b,pos)) is not None]
    if not interior:return None,None
    d,pid=min(interior,key=lambda x:x[0]);return d,pid

def candidates(r):
    carrier=r.get('carrier_xy')
    if not valid(carrier):return []
    opponents=actors(r,'opponents',True)
    detected_opponents=actors(r,'opponents',False)
    result=[]
    for pid,target,quality in actors(r,'teammates',True):
        separation=distance(carrier,target)
        d,defender=clearance(carrier,target,opponents)
        d_det,defender_det=(clearance(carrier,target,detected_opponents)
                             if r.get('carrier_detected') is True and quality=='detected'
                             else (None,None))
        result.append({
           'match_id':r['match_id'],'period':r['period'],'frame':r['frame'],
           'possession_event_id':r.get('possession_event_id'),
           'carrier_player_id':r['carrier_player_id'],'teammate_player_id':pid,
           'carrier_detected':r.get('carrier_detected') is True,
           'teammate_coordinate_quality':quality,
           'pass_segment_length_m':separation,
           'opponents_with_finite_xy':len(opponents),
           'interior_opponent_clearance_m':d,
           'closest_interior_opponent_id':defender,
           'interior_detected_only_clearance_m':d_det,
           'closest_interior_detected_opponent_id':defender_det,
           'phase_status':r.get('phase_status'),
           'geometry_only':True})
    return result


## Produce candidate table and coverage gate

If a defender projects outside the *interior* of the segment, the interior-clearance field remains `null`. This is **not** a claim that the route is open; endpoint defender proximity, ball trajectory, speed, and reaction times are not yet modelled. Counts with no interior projection are reported separately from missing opponent coverage.

In [3]:
reports=[]
for file in FILES:
    mid=file.name.split('_match_')[1].split('_ROLES')[0]
    count=Counter();examples=[]
    target=OUT/f'P2_pass_lane_candidates_match_{mid}.jsonl'
    with file.open(encoding='utf-8') as stream,target.open('w',encoding='utf-8') as out:
        for line in stream:
            row=json.loads(line);count['decision_rows']+=1
            if not valid(row.get('carrier_xy')):
                count['missing_carrier_xy']+=1;continue
            count['carrier_xy_valid']+=1
            if row.get('carrier_detected') is True:count['carrier_detected']+=1
            opp=actors(row,'opponents',True)
            if not opp:count['no_opponent_positions']+=1
            items=candidates(row)
            if not items:count['decisions_without_teammate_candidates']+=1
            for item in items:
                count['candidate_pairs']+=1
                count['teammate_detected_pairs']+=int(item['teammate_coordinate_quality']=='detected')
                count['interior_clearance_defined']+=int(item['interior_opponent_clearance_m'] is not None)
                count['detected_only_interior_clearance_defined']+=int(item['interior_detected_only_clearance_m'] is not None)
                out.write(json.dumps(item,separators=(',',':'))+'\n')
                if len(examples)<3:examples.append({k:item[k] for k in (
                    'period','frame','carrier_player_id','teammate_player_id',
                    'pass_segment_length_m','interior_opponent_clearance_m','teammate_coordinate_quality')})
    reports.append({'match_id':mid,**dict(count),'sample_candidates':examples})
    print(mid,'decision rows',count['decision_rows'],'candidate pairs',count['candidate_pairs'])

1874553 decision rows 959 candidate pairs 9560
1886347 decision rows 999 candidate pairs 9990
1899585 decision rows 929 candidate pairs 9290
1925299 decision rows 1062 candidate pairs 10620
1927964 decision rows 1140 candidate pairs 11390
1953632 decision rows 979 candidate pairs 9700
1959846 decision rows 946 candidate pairs 9450
1986691 decision rows 915 candidate pairs 9150
1996435 decision rows 1005 candidate pairs 9970
1996436 decision rows 786 candidate pairs 7850
2006229 decision rows 979 candidate pairs 9770
2006363 decision rows 976 candidate pairs 9760
2007448 decision rows 863 candidate pairs 8620
2007721 decision rows 959 candidate pairs 9590
2010085 decision rows 935 candidate pairs 9350
2011166 decision rows 757 candidate pairs 7550
2013725 decision rows 922 candidate pairs 9170
2015213 decision rows 883 candidate pairs 8810
2016236 decision rows 959 candidate pairs 9590
2017461 decision rows 839 candidate pairs 8390


In [4]:
metrics=['decision_rows','carrier_xy_valid','carrier_detected','missing_carrier_xy',
 'no_opponent_positions','decisions_without_teammate_candidates','candidate_pairs',
 'teammate_detected_pairs','interior_clearance_defined','detected_only_interior_clearance_defined']
summary={key:sum(r.get(key,0) for r in reports) for key in metrics}
assert summary['decision_rows']==18792, f'Unexpected decision count: {summary["decision_rows"]}'
report={'created_utc':datetime.now(timezone.utc).isoformat(),
 'purpose':'Static pre-action carrier-to-teammate segment geometry, not passing feasibility',
 'n_matches':len(reports),'totals':summary,'per_match':reports,
 'interpretation_constraints':[
  'One row per candidate teammate at observed carrier possession-start frame',
  'Interior-clearance undefined does NOT mean unobstructed or successful pass',
  'No speed, defender reaction time, passing angle, ball motion, pitch orientation or path dynamics',
  'Detected-only comparison is incomplete when carrier, teammate or defenders not detected',
  'No outcome labels used as spatial predictors; no pass/carry opportunity classification',
  'No predictive validation or claims about coordinated pressing']}
path=OUT/'P2_pass_lane_geometry_gate.json';path.write_text(json.dumps(report,indent=2))
print(json.dumps(summary,indent=2));print('Upload gate:',path)

{
  "decision_rows": 18792,
  "carrier_xy_valid": 18757,
  "carrier_detected": 18092,
  "missing_carrier_xy": 35,
  "no_opponent_positions": 0,
  "decisions_without_teammate_candidates": 0,
  "candidate_pairs": 187570,
  "teammate_detected_pairs": 119042,
  "interior_clearance_defined": 183324,
  "detected_only_interior_clearance_defined": 114886
}
Upload gate: /Users/fadihaddad/Documents/GitHub/SkillCorner_opendata_soccer/outputs/P2_pass_lane_geometry_gate.json
